[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagedSaeed/instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3,4"

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [3]:
# !echo $GT_TOKEN # makesure the token is loaded

In [4]:
if 'jrcai_corekit' not in os.listdir('.'):
    !git clone https://${GT_TOKEN}@github.com/MagedSaeed/jrcai_corekit.git
else: # else, pull latest changes
    !cd jrcai_corekit && git pull && cd ..

Already up to date.


In [5]:
!pip install -r jrcai_corekit/requirements.txt

/bin/bash: /home/majed_alshaibani/Projects/instructions-tuning/venv/bin/pip: /home/majed_alshaibani/Projects/InstructionsTuning/venv/bin/python3: bad interpreter: No such file or directory


add jrcai_corekit to path

In [6]:
import sys
sys.path.append('jrcai_corekit/src')

check everything is working

In [7]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


# Constants

In [8]:
TAWJEEH_DATASET_NAME = 'xlsum'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/xlsum_arabic_experimental'
TASK_NAME='summarization'
MODEL_PATH = "/hdd/shared_models/Meta-Llama-3.1-8B"

In [9]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [10]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://tawjeeh.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14888,
  'tags': ['AI generated'],
  'name': 'history_based_dialect',
  'task': {'name': 'dialect identification'},
  'status': 'SUBMITTED',
  'template': 'Consider this historical context in the text provided: {{Text}}. Which dialect from the ancient regions does it emerge from? Select from these regions: Levant, North Africa, Egypt, GULF, MSA. ||| {{answer_choices[label]}}',
  'dataset_name': 'arbml/Arabic_Dialects_Dataset',
  'dataset_subset': 'default',
  'answer_choices': ['Levant', 'North Africa', 'Egypt', 'GULF', 'MSA'],
  'text_direction': 'ltr'},
 {'id': 14887,
  'tags': ['AI generated'],
  'name': 'literary_style_identification',
  'task': {'name': 'dialect identification'},
  'status': 'SUBMITTED',
  'template': 'The narrative style found in the following phrase: {{Text}}, is tied to a specific dialect. Out of Levant, North Africa, Egypt, GULF, MSA, which one do you think it is? ||| {{answer_choices[label]}}',
  'dataset_name': 'arbml/Arabic_Dialects_Dataset',
  'dat

In [11]:
len(prompts)

351

In [12]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

223

## Finetuning

### Get the dataset prompts

In [13]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

7

In [14]:
SELECTED_PROMPTS_IDS = [
    14871,   
    14803,
    14856,
    14858,
    14668,
]

In [15]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [16]:
import datasets

In [17]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['gem_id', 'url', 'title', 'target', 'references', 'text'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['gem_id', 'url', 'title', 'target', 'references', 'text'],
        num_rows: 4689
    })
})

### Merge the prompts

In [18]:
from jinja2 import Environment, StrictUndefined

In [19]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    prefix = prefix.replace('\xa0','')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}|||{suffix.strip()}'

In [20]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [21]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][-700]))

You are an Arabic summarization expert! The summarization of the following Arabic passage: "(أرشيف) قالت السلطات المصرية إن الجيش والشرطة اللذين يشنان حملة على الجماعة قتلا مئات من أعضائها.

ونشرت الجماعة على موقع تويتر مقطعا مصورا يظهر عشرات القتلى من الضباط والجنود في الهجوم الذي وقع يوم 24 أكتوبر/ تشرين الأول مستهدفا نقطة التفتيش العسكرية في منطقة كرم القواديس قرب مدينة الشيخ زويد بشمال سيناء.

وظهر في الشريط المصور رجل يحذر الرئيس المصري عبد الفتاح السيسي من هجمات سوف تشنها الجماعة على قوات الجيش والشرطة في شمال سيناء.

وكتب في شريط في أسفل الشاشة "الاستشهادي أبو حمزة الأنصاري تقبله الله الغائر على نقطة كرم القواديس العسكرية."

لكن الشريط المصور لم يتضمن تاريخ الهجوم على النقطة ولم يتسن التحقق من صحته حتى الآن.

مواضيع قد تهمك نهاية

وأطلقت الجماعة المتشددة على نفسها اسم ولاية سيناء المصرية بعد أن أعلنت الخميس مبايعة زعيم تنظيم الدولة الاسلامية أبو بكر البغدادي.

وكانت جماعة أنصار بيت المقدس، وهي أكبر جماعة إسلامية متشددة في مصر، أعلنت انضمامها إلى تنظيم الدولة الإسلامية الذى استول

In [22]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

6000.0

In [23]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/30000 [00:00<?, ?it/s]

rending You are an Arabic summarization expert! The summarization of the following Arabic passage: "{{text}}" is:
|||
{{target}} sample index: 0
rending Generate a summary for the following Arabic document: {{text}}
|||
{{target}} sample index: 6000
rending A concise and informative summary that captures the main points of the Arabic article "{{text}}" while maintaining its original meaning is:
|||
{{target}} sample index: 12000
rending Create tldr for the following text: {{text}}
|||
{{target}} sample index: 18000
rending This article {{text}}  with the title {{title}}  can be summarized as:
|||
{{target}} sample index: 24000


30000

## Finetune the LLM

In [24]:
GLOBAL_SEED = 42

In [25]:
import random
random.seed(GLOBAL_SEED)

In [26]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Llama3Initializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [27]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Llama3Initializer(),
)
llm_loader

In [28]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/config.json
Model config LlamaConfig {
  "_name_or_path": "/hdd/shared_models/Meta-Llama-3.1-8B",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 128256
}


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing LlamaForCausalLM.

All the weights of LlamaForCausalLM were initialized from the model checkpoint at /hdd/shared_models/Meta-Llama-3.1-8B.
If your task is similar to the task the model of the checkpoint was trained on, you can already use LlamaForCausalLM for predictions without further training.
loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.9
}

loading file tokenizer.json
loading file tokenizer.model
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/generation_config.json
Generate config GenerationConfig {
  "bos_tok

In [29]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    prefix,suffix = sample.split('|||')
    prefix = prefix.strip()
    prefix = prefix.replace('\xa0','')
    suffix = suffix.strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(27000,
 3000,
 [('You are an Arabic summarization expert! The summarization of the following Arabic passage: "زعماء الترويكا الحاكمة في تونس يحتفلون بإقرار المجلس التأسيسي أول دستور بعد الثورة\n\n واوضح التصويت على الدستور الجديد وجود درجة عالية من التوافق على بنوده، اذ وافق عليه 200 عضو من اعضاء المجلس التأسيسي، والذين يبلغ عددهم 216 عضوا. وبشكل عام ينظر الى اقرار الدستور على انه خطوة سياسية هامة على طريق تحقيق الاستقرار وبناء المؤسسات في تونس. وتزامن مع اقرار الدستور الاتفاق على تشكيل حكومة جديدة من شخصيات مستقلة برئاسة مهدي جمعة. وكان الرئيس التونسي المنصف المرزوقي وصف تجربة بلاده "بالمعجزة التونسية"، اذ تمكنت تونس "من الحفاظ على الحرية والامن ونموذج من الاعتدال" على حد وصفه.\n\n غير ان هناك تحديات هائلة لازالت تواجه تونس، وتتطلب عملا شاقا للتعامل معها وتجاوزها. فعلى المستوى الاقتصادي يعاني الشباب التونسي من بطالة واسعة، ويعاني المجتمع كله من ارتفاع كبير في الاسعار، وهناك ضغوط لمواجهة العجز في موازنة الدولة، والتي ترجع في جزء اساسي منها الى الدعم الحكومي لمجموعة من السلع الغذائية و

In [30]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=1,
    eval_batch_size=1,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}',
    early_stopping_patience=20,
    eval_steps=1000,
)

peft config LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj'}, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto half precision backend

***** Running Evaluation *****
  Num examples = 3000
  Batch size = 1


loading configuration file /hdd/shared_models/Meta-Llama-3.1-8B/config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 128256
}



{'eval_loss': 1.6897685527801514, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 335.9839, 'eval_samples_per_second': 8.929, 'eval_steps_per_second': 8.929}


***** Running training *****
  Num examples = 27,000
  Num Epochs = 10
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 1
  Gradient Accumulation steps = 1
  Total optimization steps = 270,000
  Number of trainable parameters = 6,815,744


...

{'eval_loss': 1.4506425857543945, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 329.2277, 'eval_samples_per_second': 9.112, 'eval_steps_per_second': 9.112, 'epoch': 0.7407407407407407}



***** Running Evaluation *****
  Num examples = 3000
  Batch size = 1


{'eval_loss': 1.4463688135147095, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 330.0266, 'eval_samples_per_second': 9.09, 'eval_steps_per_second': 9.09, 'epoch': 0.7777777777777778}




Training completed. Do not forget to share your model on huggingface.co/models =)




1.4323056936264038

In [ ]:
exit()

: 